### Conversie taaksjabloon (word -> json)

In [1]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
import pprint

paths = read_dir('/Users/baasa03/Digicoach/taaksjabloon_demo')
json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

LIC - Taaksjablonen Sanering v1.0.docx



retrieved 9 tables from doc /Users/baasa03/Digicoach/taaksjabloon_demo/LIC - Taaksjablonen Sanering v1.0.docx
Found title: Sanering (LIC)
Found tag: Directie CAP
Found tag: Keten Inning en betalingsverkeer
Found step: Open postbak in WAB
Found step: Selecteer alleen kwijtscheldings / saneringsverzoeken
Found step: In behandeling nemen poststuk saneringen
Found step: Controleer taken op je naam
Found task: Selecteer Zaak
Found step: Check op vermogensbestanddelen
Found step: Check op mogelijkheid aansprakelijk stellen
Found step: Check op teruggaven en goeder trouw
Found step: Check op ambtshalve aanslagen
Found step: Vastleggen acties
Found task: Quick scan verzoek
Found step: Controleer competentie
Found step: Controleer de datum van binnenkomst
Found step: Check op benodigde stukken
Found step: Check op onbehandelde zaken
Found step: Controleer volledigheid verzoek
Found step: Vastleggen brief of actie in INL
Found step: Acties bij verzoek is

### Naamgeving digicoach

In [2]:
import json
from ask_delphi_api.import_digicoach import Import
import datetime

json_digicoach = json.loads(json_coaches[0])
naam_digicoach = f"Digicoach {json_digicoach['name']} {datetime.datetime.now()}"
tags= json_digicoach['tags']
name_voorgedefinieerde_zoekopdracht = ""
for tag in tags:
    if tag["type"] == "Keten" or tag["type"] == "Middel":
        name_voorgedefinieerde_zoekopdracht = tag["values"][0]

print(naam_digicoach)
print(name_voorgedefinieerde_zoekopdracht)

AskDelphi Client loaded.
Digicoach Sanering (LIC) 2026-02-25 11:33:01.028931
Inning en betalingsverkeer


### Bronnen uit taaksjabloon registreren als topic

In [3]:
import_service = Import()

# Ophalen bronnen lijst taaksjabloon
sources = json_digicoach["sources"]

# Aanmaken sources
import_service.create_source_topics(sources)

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!
Gevonden : 15a88ae8-e031-4612-8c1c-00c188ff600d, Kwijtscheldingsformulier Ondernemers, Externe link, https://www.belastingdienst.nl/wps/wcm/connect/bldcontentnl/themaoverstijgend/programmas_en_formulieren/kwijtschelding_van_belasting_enof_premie_voor_ondernemingen
Gevonden : 7ab88b6a-a796-40af-9fd0-680edb57a9af, Applicatie overzicht, Externe link, https://connectpeople.belastingdienst.nl/files/form/anonymous/api/library/1fa8a980-ab6d-4c9a-a059-da1fea2c3dbf/document/1a372d7d-d07f-448a-8f2b-efa5c9690ec8/media/6980%20LIC%20-%20applicatie%20overzicht.pdf
Gevonden : 43a44be0-2b2f-424a-b26d-91cf878fe08f, GDC - Standaardzinnen voor de sanering, Externe link, Word document (zelfde link als in de DC Sanering (Taskforce)
Gevonden : 41d4d431-79db-4e22-a265-245a61ef119b, LIC - memo - tijdelijke instructie 

### Creatie voorgedefinieerde_zoekopdracht

In [ ]:
from ask_delphi_api.import_digicoach import Import

topic_id_list = []


topic_id_predefined_search, topic_version_id_predefined_search = import_service.create_voorgedefinieerde_zoekopdracht_topic(name_voorgedefinieerde_zoekopdracht)
topic_id_list.append(topic_id_predefined_search)

### Creatie Digicoach

In [ ]:
topic_id_digicoach, topic_version_id_digicoach = import_service.create_digicoach(naam_digicoach, topic_id_predefined_search, topic_version_id_predefined_search)
topic_id_list.append(topic_id_digicoach)

### Tag Digicoach

In [ ]:
# Voeg tags toe
for tag in tags:
    # import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, tag["values"][0])
    import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, "interactie")
    # import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, "CAP")

### Creatie taken, stappen en publiceren

In [ ]:
# Haal takenlijst uit json_digicoach
taken = json_digicoach["tasks"]

# Voor elke taak, maak topic
for taak in taken:

    # Maak topic met naam taak uit json
    topic_id_task, topic_version_id_task = import_service.create_task(taak["name"], topic_id_digicoach, topic_version_id_digicoach)
    topic_id_list.append(topic_id_task)

    # Voeg beschrijving als content toe
    import_service.add_content_to_topic(topic_id_task, topic_version_id_task, taak['description'])

    # Haal stappenlijst uit taak
    steps = taak["steps"]

    for step in steps:
    # Maak topic met naam step uit json
        topic_id_step, topic_version_id_step = import_service.create_step(step["name"], topic_id_task, topic_version_id_task)
        topic_id_list.append(topic_id_step)

        # Toevoegen beschrijving aan de topic content
        import_service.add_content_to_topic(topic_id_step, topic_version_id_step, step['description'])

        # Aanwezige links toevoegen aan pyramide
        import_service.add_sources(topic_id_task, topic_version_id_task, step['description'], sources)

        import_service._get_topic().checkin(topic_id_step)

    # Inchecken taak
    import_service._get_topic().checkin(topic_id_task)

import_service._get_topic().checkin(topic_id_digicoach)
import_service._get_topic().checkin(topic_id_predefined_search)

for topic_id in topic_id_list:
    import_service.publiceer(topic_id)
